In [1]:

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected")


CUDA available: True
GPU: Tesla T4
GPU count: 2


In [2]:
from pathlib import Path

for file in Path("/kaggle/input").rglob("*"):
    if file.is_file():
        print(file)

/kaggle/input/datasets/arifengine/firesafe-v3-2-dataset/README.txt
/kaggle/input/datasets/arifengine/firesafe-v3-2-dataset/firesafe_v32_new_20_REVIEW_REQUIRED.jsonl
/kaggle/input/datasets/arifengine/firesafe-v3-2-dataset/firesafe_v32_page_heldout_10_REVIEW_REQUIRED.jsonl
/kaggle/input/datasets/arifengine/firesafe-v3-2-dataset/firesafe_v32_train_79_CANDIDATE.jsonl
/kaggle/input/datasets/arifengine/firesafe-v3-2-dataset/firesafe_v3_sofie_heldout_UNCHANGED.jsonl
/kaggle/input/datasets/arifengine/firesafe-v3-2-dataset/manifest.json
/kaggle/input/datasets/arifengine/new-v3-1-firesafe-dataset/README.txt
/kaggle/input/datasets/arifengine/new-v3-1-firesafe-dataset/audit_summary.json
/kaggle/input/datasets/arifengine/new-v3-1-firesafe-dataset/firesafe_v31_candidate_59_UNREVIEWED.jsonl
/kaggle/input/datasets/arifengine/new-v3-1-firesafe-dataset/firesafe_v31_new_source_grounded_40.jsonl
/kaggle/input/datasets/arifengine/new-v3-1-firesafe-dataset/firesafe_v3_heldout_eval_UNCHANGED.jsonl
/kaggle/in

In [3]:

import json
from pathlib import Path

DATA = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/data"
)

files = [
    "firesafe_v3_evidence_train.jsonl",
    "firesafe_v3_multistudy_drafts.jsonl",
    "firesafe_v3_bre_drafts.jsonl",
    "firesafe_v3_cross_study_drafts.jsonl"
]

records = []

for filename in files:
    path = DATA / filename

    with path.open(encoding="utf-8") as f:
        examples = [
            json.loads(line)
            for line in f if line.strip()
        ]

    print(f"✅ {filename}: {len(examples)}")
    records.extend(examples)

eval_path = DATA / "firesafe_v3_heldout_eval.jsonl"

with eval_path.open(encoding="utf-8") as f:
    eval_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

print("\nTraining examples:", len(records))
print("Held-out evaluation examples:", len(eval_records))

assert len(records) == 19
assert len(eval_records) == 5

print("✅ Dataset verified")


✅ firesafe_v3_evidence_train.jsonl: 4
✅ firesafe_v3_multistudy_drafts.jsonl: 5
✅ firesafe_v3_bre_drafts.jsonl: 5
✅ firesafe_v3_cross_study_drafts.jsonl: 5

Training examples: 19
Held-out evaluation examples: 5
✅ Dataset verified


In [4]:

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("GPU unavailable — training cannot start")


CUDA available: True
GPU: Tesla T4
GPU count: 2


In [5]:

from unsloth import FastLanguageModel

print("✅ Unsloth imported successfully!")


ModuleNotFoundError: No module named 'unsloth'

In [ ]:

import torch
from unsloth import FastLanguageModel

MAX_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_LEN,
    dtype=torch.float16,
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407
)

print("✅ Qwen2.5-3B loaded")
print("✅ Fresh LoRA adapter attached")


In [ ]:

import json
from pathlib import Path
from datasets import Dataset

DATA = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/data"
)

filenames = [
    "firesafe_v3_evidence_train.jsonl",
    "firesafe_v3_multistudy_drafts.jsonl",
    "firesafe_v3_bre_drafts.jsonl",
    "firesafe_v3_cross_study_drafts.jsonl"
]

records = []

for name in filenames:
    with (DATA / name).open(encoding="utf-8") as f:
        records.extend(
            json.loads(line)
            for line in f if line.strip()
        )

# Explicitly acknowledge this is an unapproved pilot.
ALLOW_DRAFT_PILOT = True

if not ALLOW_DRAFT_PILOT:
    assert all(
        r["review_status"] == "approved"
        for r in records
    )

dataset_rows = []

for record in records:
    messages = record["messages"]

    prompt_messages = [
        m for m in messages
        if m["role"] in ["system", "user"]
    ]

    answer = next(
        m["content"]
        for m in messages
        if m["role"] == "assistant"
    )

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    completion = answer + tokenizer.eos_token

    token_count = len(
        tokenizer(
            prompt + completion,
            add_special_tokens=False
        )["input_ids"]
    )

    if token_count > MAX_LEN:
        raise ValueError(
            f"Training example exceeds {MAX_LEN} tokens: "
            f"{token_count}. Review its evidence length."
        )

    dataset_rows.append({
        "prompt": prompt,
        "completion": completion
    })

train_dataset = Dataset.from_list(dataset_rows)

print("✅ Training examples:", len(train_dataset))
print("✅ Prompt/completion formatting ready")
print("✅ Pilot mode: unapproved training drafts")


In [ ]:

from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/firesafe_qwen3b_v3_pilot"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_length=MAX_LEN,
        completion_only_loss=True,
        packing=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=5e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        fp16=True,
        bf16=False,
        logging_steps=1,
        save_strategy="epoch",
        report_to="none",
        seed=3407
    )
)

# Check that only completion tokens receive training loss.
batch = next(iter(trainer.get_train_dataloader()))
labels = batch["labels"]

active = int((labels != -100).sum().item())
masked = int((labels == -100).sum().item())

print("Tokens used for loss:", active)
print("Masked tokens:", masked)

assert active > 0, "No answer tokens selected"
assert masked > 0, "Prompt masking is not working"

print("✅ Assistant-answer training verified")

trainer.train()

print("✅ FireSafe AI v3 pilot training completed")


In [ ]:

from pathlib import Path
import shutil

adapter_dir = Path(
    "/kaggle/working/firesafe_qwen3b_v3_adapter"
)

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

zip_path = shutil.make_archive(
    "/kaggle/working/FireSafe_Qwen3B_v3_Pilot",
    "zip",
    root_dir=adapter_dir
)

print("✅ Saved:", zip_path)


In [ ]:

import json
from pathlib import Path

DATA = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/data"
)

eval_path = DATA / "firesafe_v3_heldout_eval.jsonl"

with eval_path.open(encoding="utf-8") as f:
    eval_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

print("✅ Evaluation questions loaded:", len(eval_records))


In [ ]:

import json
import torch
from pathlib import Path
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

evaluation_results = []

for record in eval_records:
    messages = record["messages"]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    evaluation_results.append({
        "id": record["id"],
        "question": messages[-1]["content"].split(
            "\n\nEvidence:"
        )[0],
        "qwen_v3_answer": answer,
        "reference_answer": record["reference_answer"]
    })

    print("\n" + "=" * 65)
    print("QUESTION:", evaluation_results[-1]["question"])
    print("\nQWEN V3 ANSWER:\n", answer)
    print("\nREFERENCE ANSWER:\n", record["reference_answer"])

output = Path(
    "/kaggle/working/firesafe_v3_evaluation_results.json"
)

output.write_text(
    json.dumps(
        evaluation_results,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n✅ Evaluation completed:", len(evaluation_results), "questions")
print("✅ Results saved:", output)


In [ ]:

import json
import torch
from pathlib import Path
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

STRICT_RULES = """You are FireSafe AI, a scientific research assistant.

Answer the user's specific question using ONLY the supplied evidence.

Rules:
1. Distinguish research objectives, hypotheses, and measured findings.
2. Never describe an objective or expected outcome as an observed result.
3. Do not invent measurements, comparisons, or experimental conclusions.
4. Cite supported statements using the source identifier in the evidence,
   e.g. [SOFIE-MIST:NASA].
5. If measurements are not reported, explicitly say so.
6. Answer concisely. Do not add sections the question did not request.
7. Do not infer habitat safety from a general experiment description.
"""

strict_results = []

for record in eval_records:
    original = record["messages"]

    messages = [
        {"role": "system", "content": STRICT_RULES}
    ] + [
        m for m in original
        if m["role"] != "system"
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    strict_results.append({
        "id": record["id"],
        "answer": answer
    })

    print("\n" + "=" * 60)
    print("QUESTION:", messages[-1]["content"].splitlines()[0])
    print("NEW ANSWER:", answer)

path = Path(
    "/kaggle/working/firesafe_v3_strict_eval.json"
)

path.write_text(
    json.dumps(strict_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ Strict evaluation saved:", path)


In [ ]:

import gc
import torch
from unsloth import FastLanguageModel

# Release the fine-tuned model from GPU memory.
if "trainer" in globals():
    del trainer

del model
gc.collect()
torch.cuda.empty_cache()

# Load a clean base model WITHOUT attaching the v3 adapter.
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True
)

FastLanguageModel.for_inference(base_model)

print("✅ Original Qwen2.5-3B loaded")
print("✅ No v3 adapter attached")


In [ ]:

import json
from pathlib import Path

STRICT_RULES = """You are FireSafe AI, a scientific research assistant.

Answer the user's specific question using ONLY the supplied evidence.

Rules:
1. Distinguish research objectives, hypotheses, and measured findings.
2. Never describe an objective or expected outcome as an observed result.
3. Do not invent measurements, comparisons, or experimental conclusions.
4. Cite supported statements using the source identifier in the evidence,
   e.g. [SOFIE-MIST:NASA].
5. If measurements are not reported, explicitly say so.
6. Answer concisely. Do not add sections the question did not request.
7. Do not infer habitat safety from a general experiment description.
"""

# Load the exact earlier v3 responses for comparison.
v3_path = Path("/kaggle/working/firesafe_v3_strict_eval.json")
v3_results = json.loads(v3_path.read_text(encoding="utf-8"))
v3_by_id = {r["id"]: r["answer"] for r in v3_results}

comparison = []

for record in eval_records:
    messages = [
        {"role": "system", "content": STRICT_RULES}
    ] + [
        m for m in record["messages"]
        if m["role"] != "system"
    ]

    prompt = base_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda:0")

    with torch.inference_mode():
        output = base_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id
        )

    base_answer = base_tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    result = {
        "id": record["id"],
        "question": messages[-1]["content"].split("\n\nEvidence:")[0],
        "reference_answer": record["reference_answer"],
        "base_qwen": base_answer,
        "fine_tuned_v3": v3_by_id.get(record["id"], "MISSING")
    }

    comparison.append(result)

    print("\n" + "=" * 70)
    print("QUESTION:", result["question"])
    print("\nBASE QWEN:", result["base_qwen"])
    print("\nFINE-TUNED V3:", result["fine_tuned_v3"])
    print("\nREFERENCE:", result["reference_answer"])

output_path = Path(
    "/kaggle/working/firesafe_v3_vs_base_comparison.json"
)

output_path.write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ A/B comparison completed")
print("✅ Results:", output_path)


In [ ]:
 
from pathlib import Path
import re
import fitz  # PyMuPDF
from sklearn.feature_extraction.text import TfidfVectorizer

PAPERS = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/papers"
)

paper_files = {
    "SAFFIRE": "saffire_research.pdf",
    "BASS": "BASS-20140011099.pdf",
    "BASSII": "BASSII-20160000593.pdf",
    "BRE": "BRE-20160010073.pdf",
    "ACME-BRE": "ACME-BRE-OVERVIEW.pdf",
}

chunks = []

for source, filename in paper_files.items():
    path = PAPERS / filename
    assert path.exists(), f"Missing PDF: {path}"

    with fitz.open(path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text")
            text = re.sub(r"\s+", " ", text).strip()
            words = text.split()

            # Overlapping windows preserve some context.
            chunk_size = 180
            overlap = 45
            stride = chunk_size - overlap

            for start in range(0, len(words), stride):
                part = words[start:start + chunk_size]

                # Skip very short fragments.
                if len(part) < 35:
                    continue

                chunks.append({
                    "source": source,
                    "filename": filename,
                    "pdf_page": page_number,
                    "citation": f"{source}:pdf-p{page_number}",
                    "text": " ".join(part)
                })

print("✅ NASA papers indexed:", len(paper_files))
print("✅ Searchable passages:", len(chunks))

# Hybrid word and character TF-IDF improves lexical matching
# without requiring additional model downloads.
word_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    sublinear_tf=True
)

texts = [c["text"] for c in chunks]

word_matrix = word_vectorizer.fit_transform(texts)
char_matrix = char_vectorizer.fit_transform(texts)

print("✅ PDF retrieval index ready")


In [ ]:

import numpy as np

def retrieve_evidence(question, top_k=5):
    word_query = word_vectorizer.transform([question])
    char_query = char_vectorizer.transform([question])

    word_scores = (
        word_matrix @ word_query.T
    ).toarray().ravel()

    char_scores = (
        char_matrix @ char_query.T
    ).toarray().ravel()

    scores = 0.7 * word_scores + 0.3 * char_scores

    ranked = np.argsort(scores)[::-1]

    selected = []
    seen_pages = set()

    for index in ranked:
        if scores[index] <= 0:
            break

        chunk = chunks[int(index)]
        page_key = (
            chunk["source"],
            chunk["pdf_page"]
        )

        # Prevent several overlapping snippets from the same page.
        if page_key in seen_pages:
            continue

        seen_pages.add(page_key)

        selected.append({
            **chunk,
            "score": float(scores[index])
        })

        if len(selected) >= top_k:
            break

    return selected


def show_evidence(question, top_k=5):
    results = retrieve_evidence(question, top_k)

    print("QUESTION:", question)

    for i, item in enumerate(results, start=1):
        print("\n" + "=" * 65)
        print(
            f"{i}. [{item['citation']}] "
            f"{item['filename']}"
        )
        print("Retrieval score:", round(item["score"], 3))
        print(item["text"][:750] + "...")

    return results


test_question = (
    "How does airflow velocity affect "
    "flame spread in microgravity?"
)

evidence = show_evidence(test_question)


In [ ]:

import json
from pathlib import Path

SYSTEM = """You are a NASA fire research assistant.
Answer only from the supplied passages.
Preserve the distinction between experiments, observations,
hypotheses, model-based explanations, and limitations.
Do not invent numbers, causes, thresholds, or conclusions.
Cite only the supplied page identifiers.
Answer the question directly."""

QUESTION = (
    "How does airflow velocity affect flame spread "
    "in microgravity according to NASA experiments?"
)

def run_focused_test(test_model, test_tokenizer):
    import torch

    messages = [
        {"role": "system", "content": SYSTEM},
        {
            "role": "user",
            "content": (
                f"Question: {QUESTION}\n\n"
                f"NASA passages:\n{evidence_text}"
            )
        }
    ]

    prompt = test_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda:0")

    if inputs["input_ids"].shape[1] > 3500:
        raise ValueError("Evidence too long for fair comparison")

    with torch.inference_mode():
        output = test_model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=False,
            pad_token_id=test_tokenizer.eos_token_id
        )

    return test_tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

v3_answer = run_focused_test(model, tokenizer)

print("FINE-TUNED V3:\n")
print(v3_answer)

Path("/kaggle/working/v3_focused_answer.txt").write_text(
    v3_answer, encoding="utf-8"
)


In [ ]:

import gc
import torch
from unsloth import FastLanguageModel

model.save_pretrained(
    "/kaggle/working/firesafe_qwen3b_v3_adapter"
)
tokenizer.save_pretrained(
    "/kaggle/working/firesafe_qwen3b_v3_adapter"
)

del model
gc.collect()
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True
)

FastLanguageModel.for_inference(base_model)

base_answer = run_focused_test(
    base_model, base_tokenizer
)

print("=" * 65)
print("ORIGINAL QWEN:\n", base_answer)
print("\n" + "=" * 65)
print("FINE-TUNED V3:\n", v3_answer)

comparison = {
    "question": QUESTION,
    "base_answer": base_answer,
    "v3_answer": v3_answer,
    "evidence_pages": [
        e["citation"] for e in focused_evidence
    ]
}

Path(
    "/kaggle/working/firesafe_focused_ab_test.json"
).write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8"
)


In [ ]:

import json
from pathlib import Path

fix_ids = {
    "v3_bre_method",
    "v3_bass_bre_comparison"
}

fixed_count = 0

for record in records:
    if record["example_id"] not in fix_ids:
        continue

    for message in record["messages"]:
        message["content"] = message["content"].replace(
            "[BRE:p2]", "[BREOV:p2]"
        )

    fixed_count += 1

assert fixed_count == 2

# Verify citations against declared sources.
import re

issues = []

for record in records:
    answer = " ".join(
        m["content"]
        for m in record["messages"]
        if m["role"] == "assistant"
    )

    cited = set(re.findall(
        r"\[([A-Za-z0-9_-]+:p\d+)\]",
        answer
    ))

    unexpected = cited - set(record["source_pages"])

    if unexpected:
        issues.append((record["example_id"], unexpected))

assert not issues, issues

OUTPUT = Path(
    "/kaggle/working/firesafe_v31_corrected_59_CANDIDATE.jsonl"
)

with OUTPUT.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Citation labels corrected:", fixed_count)
print("✅ Unexpected citations remaining:", len(issues))
print("✅ Candidate dataset saved:", OUTPUT)


In [ ]:

import gc
import torch
from unsloth import FastLanguageModel

# Release any previously loaded models.
for name in ["model", "base_model", "trainer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

MAX_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_LEN,
    dtype=torch.float16,
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407
)

print("✅ Fresh Qwen2.5-3B model ready")
print("✅ New v3.1 LoRA adapter attached")


In [ ]:

import json
from pathlib import Path
from datasets import Dataset

DATA_FILE = Path(
    "/kaggle/working/firesafe_v31_corrected_59_CANDIDATE.jsonl"
)

with DATA_FILE.open(encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f if line.strip()
    ]

train_rows = []

for record in records:
    messages = record["messages"]

    prompt_messages = [
        m for m in messages
        if m["role"] in ("system", "user")
    ]

    answer = next(
        m["content"]
        for m in messages
        if m["role"] == "assistant"
    )

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    completion = answer + tokenizer.eos_token

    tokens = tokenizer(
        prompt + completion,
        add_special_tokens=False
    )["input_ids"]

    if len(tokens) > MAX_LEN:
        raise ValueError(
            f"{record['example_id']} exceeds "
            f"the {MAX_LEN}-token limit: {len(tokens)}"
        )

    train_rows.append({
        "prompt": prompt,
        "completion": completion
    })

train_dataset = Dataset.from_list(train_rows)

assert len(train_dataset) == 59

print("✅ 59 candidate examples prepared")
print("✅ Assistant-only training format ready")


In [ ]:

from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/firesafe_v31_checkpoints"

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_length=MAX_LEN,
        completion_only_loss=True,
        packing=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-5,
        warmup_steps=2,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        fp16=True,
        bf16=False,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=3,
        report_to="none",
        seed=3407
    )
)

# Confirm the prompt is masked and answer tokens are trained.
batch = next(iter(trainer.get_train_dataloader()))

active = int((batch["labels"] != -100).sum())
masked = int((batch["labels"] == -100).sum())

print("Answer tokens used for loss:", active)
print("Prompt tokens masked:", masked)

assert active > 0
assert masked > 0

print("✅ Assistant-only loss verified")

trainer.train()

print("✅ FireSafe AI v3.1 pilot training completed")


In [ ]:

from pathlib import Path
import shutil

adapter_dir = Path("/kaggle/working/firesafe_qwen3b_v31_adapter")

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

zip_path = shutil.make_archive(
    "/kaggle/working/FireSafe_Qwen3B_v31_Pilot",
    "zip",
    root_dir=adapter_dir
)

print("✅ FireSafe v3.1 saved:", zip_path)


In [ ]:

import re

def normalize_and_check_citations(answer, allowed_citations):
    allowed = set(allowed_citations)

    def replace_citation(match):
        cited = match.group(1)

        if cited in allowed:
            return f"[{cited}]"

        # Accept the short-form alias only if the exact
        # PDF-page citation was actually supplied.
        alias = re.fullmatch(
            r"([A-Za-z0-9_-]+):p(\d+)",
            cited
        )

        if alias:
            canonical = (
                f"{alias.group(1)}:pdf-p{alias.group(2)}"
            )

            if canonical in allowed:
                return f"[{canonical}]"

        return match.group(0)

    normalized = re.sub(
        r"\[([A-Za-z0-9_-]+:(?:pdf-)?p\d+)\]",
        replace_citation,
        answer
    )

    cited_after = set(re.findall(
        r"\[([A-Za-z0-9_-]+:(?:pdf-)?p\d+)\]",
        normalized
    ))

    return {
        "answer": normalized,
        "invalid": sorted(cited_after - allowed),
        "missing": len(cited_after) == 0
    }


for result in results:
    checked = normalize_and_check_citations(
        result["answer"],
        result["evidence_pages"]
    )

    print(result["id"])
    print("Invalid citations:", checked["invalid"])
    print("Missing citations:", checked["missing"])
    print()


In [ ]:

import gc
import torch
from unsloth import FastLanguageModel

# Keep v3.1 results, which are already stored in `results`.
v31_results = results.copy()

# Release fine-tuned model.
if "trainer" in globals():
    del trainer

del model
gc.collect()
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True
)

FastLanguageModel.for_inference(base_model)

print("✅ Original Qwen loaded without v3.1 adapter")


In [ ]:

import json
from pathlib import Path

def run_base_test(test):
    pages = [load_page(*p) for p in test["pages"]]

    evidence = "\n\n".join(
        f"[{p['citation']}]\n{p['text']}"
        for p in pages
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Question: {test['question']}\n\n"
                f"NASA evidence:\n{evidence}"
            )
        }
    ]

    prompt = base_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda:0")

    if inputs["input_ids"].shape[1] > 3600:
        raise ValueError("Prompt exceeds test context budget")

    with torch.inference_mode():
        output = base_model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id
        )

    answer = base_tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return answer


base_results = []

for test in TESTS:
    answer = run_base_test(test)

    base_results.append({
        "id": test["id"],
        "question": test["question"],
        "answer": answer
    })

    v31_answer = next(
        r["answer"]
        for r in v31_results
        if r["id"] == test["id"]
    )

    print("\n" + "=" * 70)
    print("TEST:", test["id"])
    print("\nBASE QWEN:\n", answer)
    print("\nFIRE SAFE V3.1:\n", v31_answer)

Path(
    "/kaggle/working/firesafe_base_vs_v31.json"
).write_text(
    json.dumps(
        {
            "base": base_results,
            "v31": v31_results
        },
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n✅ Base vs. v3.1 comparison saved")


In [ ]:

import json
from pathlib import Path

baseline_path = Path(
    "/kaggle/working/firesafe_v31_on_v32_heldout.json"
)

if not baseline_path.exists():
    assert "v31_baseline" in globals()
    baseline_path.write_text(
        json.dumps(v31_baseline, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

baseline = json.loads(baseline_path.read_text(encoding="utf-8"))

assert len(baseline) == 10
print("✅ v3.1 baseline securely saved:", baseline_path)


In [ ]:

import gc
import torch
from unsloth import FastLanguageModel

for name in ["model", "base_model", "trainer"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

MAX_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_LEN,
    dtype=torch.float16,
    load_in_4bit=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407
)

print("✅ Fresh Qwen2.5-3B loaded")
print("✅ New v3.2 LoRA adapter created")


In [ ]:

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir="/kaggle/working/firesafe_v32_checkpoints",
        max_length=4096,
        completion_only_loss=True,
        packing=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-5,
        warmup_steps=2,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        fp16=True,
        bf16=False,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=3,
        report_to="none",
        seed=3407
    )
)

batch = next(iter(trainer.get_train_dataloader()))

active = int((batch["labels"] != -100).sum())
masked = int((batch["labels"] == -100).sum())

print("Answer tokens:", active)
print("Masked prompt tokens:", masked)

assert active > 0 and masked > 0

trainer.train()

print("✅ FireSafe AI v3.2 training finished")


In [ ]:

from pathlib import Path
import shutil

adapter_dir = Path(
    "/kaggle/working/firesafe_qwen3b_v32_adapter"
)

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

zip_path = shutil.make_archive(
    "/kaggle/working/FireSafe_Qwen3B_v32_Pilot",
    "zip",
    root_dir=adapter_dir
)

print("✅ v3.2 saved:", zip_path)


In [ ]:

import json
from pathlib import Path
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# The same function and dataset used for v3.1.
v32_results = run_benchmark(model, tokenizer, v32_eval)

assert len(v32_results) == 10

output_path = Path(
    "/kaggle/working/firesafe_v32_on_v32_heldout.json"
)

output_path.write_text(
    json.dumps(v32_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ v3.2 evaluation completed")
print("✅ Results saved:", output_path)


In [ ]:

import json
from pathlib import Path

v31_path = Path(
    "/kaggle/working/firesafe_v31_on_v32_heldout.json"
)

v32_path = Path(
    "/kaggle/working/firesafe_v32_on_v32_heldout.json"
)

v31 = {
    x["id"]: x
    for x in json.loads(v31_path.read_text(encoding="utf-8"))
}

v32 = {
    x["id"]: x
    for x in json.loads(v32_path.read_text(encoding="utf-8"))
}

for test_id, new in v32.items():
    old = v31[test_id]

    print("\n" + "=" * 70)
    print("TEST:", test_id)
    print("QUESTION:", new["question"])
    print("\nV3.1:", old["answer"])
    print("\nV3.2:", new["answer"])
    print("\nREFERENCE:", new["reference_answer"])


In [ ]:

import json
import re
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def generate_v4(messages, max_new_tokens=500):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda:0")

    # Reserve room for the answer.
    if inputs["input_ids"].shape[1] + max_new_tokens > 4096:
        raise ValueError(
            "Evidence too long. Reduce retrieved passages."
        )

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()


def parse_json_object(text):
    # Accept a JSON object, even if the model adds a code fence.
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object in model response")

    obj, _ = json.JSONDecoder().raw_decode(text[start:])
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")

    return obj


EXTRACTION_SYSTEM = """You extract scientific evidence
from NASA research passages.

Return ONLY a JSON object with this structure:

{
  "observations": [
    {
      "experiment": "exact experiment identity",
      "measurement": "what was observed",
      "value": "reported value or null",
      "condition": "experimental conditions",
      "evidence_type": "observation or hypothesis or objective",
      "source_id": "exact supplied citation identifier",
      "support_quote": "short exact phrase from source text"
    }
  ],
  "limitations": ["limitations supported by supplied evidence"]
}

Rules:
- Extract only facts relevant to the question.
- Do not invent missing measurements.
- Keep distinct experiments in separate records.
- Do not confuse timestamps with durations.
- Distinguish observations from hypotheses.
- Use only supplied citation IDs.
- Each support_quote must occur verbatim in its source.
- If a requested value is unavailable, use null.
- Do not provide a final explanatory answer yet.
"""


def extract_v4_evidence(test):
    evidence_text = "\n\n".join(
        f"[{e['citation']}]\n{e['text']}"
        for e in test["evidence"]
    )

    raw = generate_v4([
        {"role": "system", "content": EXTRACTION_SYSTEM},
        {
            "role": "user",
            "content": (
                f"Question: {test['question']}\n\n"
                f"NASA EVIDENCE:\n{evidence_text}"
            )
        }
    ], max_new_tokens=650)

    try:
        data = parse_json_object(raw)
    except Exception:
        print("Extraction was not valid JSON:\n", raw)
        raise

    allowed = {
        e["citation"]: e["text"]
        for e in test["evidence"]
    }

    verified = []
    rejected = []

    for obs in data.get("observations", []):
        if not isinstance(obs, dict):
            rejected.append({"reason": "Invalid record", "record": obs})
            continue

        source = obs.get("source_id")
        quote = obs.get("support_quote")

        # Check that the source was actually retrieved.
        if source not in allowed:
            rejected.append({"reason": "Unknown source", "record": obs})
            continue

        # Normalize whitespace for verbatim quote matching.
        source_text = " ".join(allowed[source].split())
        quote_text = " ".join(str(quote or "").split())

        if not quote_text or quote_text not in source_text:
            rejected.append({"reason": "Unmatched source quote", "record": obs})
            continue

        verified.append(obs)

    return {
        "question": test["question"],
        "observations": verified,
        "limitations": data.get("limitations", []),
        "rejected": rejected
    }

print("✅ FireSafe v4 evidence extractor ready")


In [ ]:

import json

v4_evidence = extract_v4_evidence(h07)

print(json.dumps(
    v4_evidence,
    indent=2,
    ensure_ascii=False
))

print("\nAccepted:", len(v4_evidence["observations"]))
print("Rejected:", len(v4_evidence["rejected"]))


In [ ]:

import fitz
from pathlib import Path

saffire_path = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/papers/"
    "saffire_research.pdf"
)

with fitz.open(saffire_path) as pdf:
    page8_text = " ".join(pdf[7].get_text().split())

# Create a test copy; preserve the original benchmark.
import copy

h07_fixed = copy.deepcopy(h07)

h07_fixed["evidence"].insert(0, {
    "citation": "SAFFIRE:pdf-p8",
    "filename": "saffire_research.pdf",
    "pdf_page": 8,
    "text": page8_text
})

print("✅ Added preceding Saffire page")
print("Evidence pages:", [
    e["citation"] for e in h07_fixed["evidence"]
])


In [ ]:

import fitz
import re
import json

with fitz.open(saffire_path) as pdf:
    p8 = " ".join(pdf[7].get_text().split())
    p9 = " ".join(pdf[8].get_text().split())

# Check the identity on page 8.
iv_start = p8.find("For Saffire IV-2")
assert iv_start >= 0, "Saffire IV identity missing"

iv_context = p8[iv_start:p8.find("Table 2", iv_start)]
assert "two-sided PMMA" in iv_context

# Check the continuation on page 9.
iv_match = re.search(
    r"burning in air grew to\s+"
    r"([\d.]+)\s*cm after\s+"
    r"(\d+)\s*seconds",
    p9,
    flags=re.IGNORECASE
)

assert iv_match, "Saffire IV continuation not found"

# Check the separately identified V-3 measurement.
v_match = re.search(
    r"In Saffire V-3,.*?"
    r"in\s+([\d.]+)% oxygen.*?"
    r"grew to\s+([\d.]+)\s*cm after\s+"
    r"(\d+)\s*seconds",
    p9,
    flags=re.IGNORECASE
)

assert v_match, "Saffire V-3 measurement not found"

records_h07 = [
    {
        "experiment": "Saffire IV-2",
        "measurement": "Thick PMMA flame length",
        "value": f"{iv_match.group(1)} cm",
        "growth_time": f"{iv_match.group(2)} seconds",
        "condition": "Two-sided PMMA, burning in air",
        "evidence_type": "observation",
        "source_ids": [
            "SAFFIRE:pdf-p8",
            "SAFFIRE:pdf-p9"
        ],
        "support_fragments": [
            iv_context.strip(),
            iv_match.group(0)
        ]
    },
    {
        "experiment": "Saffire V-3",
        "measurement": "Thick PMMA flame length",
        "value": f"{v_match.group(2)} cm",
        "growth_time": f"{v_match.group(3)} seconds",
        "condition": (
            f"One-sided PMMA, "
            f"{v_match.group(1)}% oxygen"
        ),
        "evidence_type": "observation",
        "source_ids": [
            "SAFFIRE:pdf-p9"
        ],
        "support_fragments": [
            v_match.group(0)
        ]
    }
]

assert float(iv_match.group(1)) == 9.5
assert int(iv_match.group(2)) == 400
assert float(v_match.group(2)) == 11
assert int(v_match.group(3)) == 450

print(json.dumps(records_h07, indent=2))


In [ ]:

import json
from pathlib import Path

v4_results = []

for test in v32_eval:
    print("\n" + "=" * 65)
    print("TEST:", test["id"])

    try:
        extracted = extract_v4_evidence(test)
        answer = synthesize_v4(extracted)

        result = {
            "id": test["id"],
            "question": test["question"],
            "answer": answer,
            "reference_answer": test["reference_answer"],
            "accepted_records": len(extracted["observations"]),
            "rejected_records": len(extracted["rejected"]),
            "rejections": extracted["rejected"],
            "extraction_limitations": extracted["limitations"],
            "status": "completed"
        }

        print("ANSWER:", answer)
        print(
            "Accepted:", result["accepted_records"],
            "| Rejected:", result["rejected_records"]
        )

    except Exception as exc:
        result = {
            "id": test["id"],
            "question": test["question"],
            "status": "error",
            "error": str(exc)
        }

        print("ERROR:", exc)

    v4_results.append(result)

output = Path(
    "/kaggle/working/firesafe_v4_automatic_benchmark.json"
)

output.write_text(
    json.dumps(v4_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ V4 benchmark saved:", output)


In [ ]:

import json
from pathlib import Path

direct_path = Path(
    "/kaggle/working/firesafe_v32_on_v32_heldout.json"
)

direct_results = {
    item["id"]: item
    for item in json.loads(
        direct_path.read_text(encoding="utf-8")
    )
}

hybrid_results = []

for item in v4_results:
    test_id = item["id"]

    if item["status"] != "completed" or (
        item.get("accepted_records", 0) == 0
    ):
        selected_answer = direct_results[test_id]["answer"]
        method = "direct_evidence_fallback"
        warning = "Structured extraction failed; answer unverified"
    else:
        selected_answer = item["answer"]
        method = "structured_extraction"
        warning = (
            "Scientific review required"
            if test_id in NEEDS_REVIEW
            else "Automated answer; scientific support not verified"
        )

    hybrid_results.append({
        "id": test_id,
        "method": method,
        "answer": selected_answer,
        "review_status": warning
    })

    print(f"{test_id}: {method} — {warning}")

output = Path(
    "/kaggle/working/firesafe_v41_hybrid_diagnostic.json"
)

output.write_text(
    json.dumps(hybrid_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ Hybrid diagnostic saved:", output)


In [ ]:

import re

def deterministic_audit(test_id, answer):
    text = answer.lower()
    issues = []

    if test_id == "H07_SAFFIRE_THICK":
        if not re.search(r"saffire\s+iv(?:-2)?\b", text):
            issues.append("Missing Saffire IV")
        if not re.search(r"saffire\s+v(?:-3)?\b", text):
            issues.append("Missing Saffire V")
        if "9.5" not in text:
            issues.append("Missing Saffire IV flame length")
        if not re.search(r"\b400\b", text):
            issues.append("Missing Saffire IV growth time")
        if not re.search(r"\b11\b", text):
            issues.append("Missing Saffire V flame length")
        if not re.search(r"\b450\b", text):
            issues.append("Missing Saffire V growth time")

    if test_id == "H10_PREVENT_VS_EXTINGUISH":
        incorrect_denial = (
            "does not provide information" in text
            or "no information" in text
        )

        if incorrect_denial:
            issues.append(
                "Incorrectly denies available source information"
            )

        if not (
            "prevent" in text
            and (
                "extinguish" in text
                or "extinction" in text
            )
        ):
            issues.append(
                "Missing prevention-versus-extinguishment distinction"
            )

    return {
        "complete": not issues,
        "issues": issues
    }


for test_id in [
    "H07_SAFFIRE_THICK",
    "H10_PREVENT_VS_EXTINGUISH"
]:
    result = next(
        r for r in v4_results if r["id"] == test_id
    )

    print(test_id)
    print(deterministic_audit(test_id, result["answer"]))


In [ ]:

import json
from pathlib import Path

direct_results = {
    r["id"]: r
    for r in json.loads(
        Path(
            "/kaggle/working/firesafe_v32_on_v32_heldout.json"
        ).read_text(encoding="utf-8")
    )
}

v42_results = []

for structured in v4_results:
    test_id = structured["id"]
    direct = direct_results[test_id]

    candidates = []

    if structured["status"] == "completed":
        candidates.append((
            "structured",
            structured["answer"]
        ))

    candidates.append((
        "direct_fallback",
        direct["answer"]
    ))

    selected = None
    failures = []

    for method, answer in candidates:
        audit = deterministic_audit(test_id, answer)

        if audit["complete"] and not (
            method == "structured"
            and structured.get("accepted_records", 0) == 0
        ):
            selected = {
                "method": method,
                "answer": answer
            }
            break

        failures.append({
            "method": method,
            "issues": audit["issues"]
        })

    if selected:
        result = {
            "id": test_id,
            "status": "candidate_answer",
            "method": selected["method"],
            "answer": selected["answer"],
            "scientific_verification": "not_verified",
            "failed_checks": failures
        }
    else:
        result = {
            "id": test_id,
            "status": "requires_review",
            "method": None,
            "answer": None,
            "scientific_verification": "not_verified",
            "failed_checks": failures
        }

    v42_results.append(result)

    print(
        test_id,
        "|", result["status"],
        "|", result["method"]
    )

output = Path(
    "/kaggle/working/firesafe_v42_hybrid_results.json"
)

output.write_text(
    json.dumps(v42_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n✅ FireSafe v4.2 results saved:", output)


In [ ]:

for test_id in [
    "H07_SAFFIRE_THICK",
    "H10_PREVENT_VS_EXTINGUISH"
]:
    result = next(
        r for r in v42_results if r["id"] == test_id
    )

    print("\n" + "=" * 65)
    print("TEST:", test_id)
    print("STATUS:", result["status"])
    print("METHOD:", result["method"])
    print("ANSWER:", result["answer"])
    print("FAILED CHECKS:", result["failed_checks"])


In [ ]:

import re

def find_named_experiments(text):
    """Identify explicitly named experiments and Saffire flights."""
    text = text.lower()

    names = set()

    for flight in re.findall(
        r"\bsaffire\s+(i{1,3}|iv|v)(?:-\d+)?\b",
        text
    ):
        names.add("Saffire " + flight.upper())

    if re.search(r"\bbass[\s-]*ii\b", text):
        names.add("BASS-II")
    elif re.search(r"\bbass\b", text):
        names.add("BASS")

    if re.search(r"\bbre\b", text):
        names.add("BRE")

    return sorted(names)


def generic_coverage_check(question, answer, evidence):
    requested = find_named_experiments(question)
    mentioned = find_named_experiments(answer)

    missing = sorted(set(requested) - set(mentioned))
    issues = []

    for experiment in missing:
        issues.append(
            f"Question mentions {experiment}, "
            "but answer does not"
        )

    # Detect unsupported claims that the source says nothing
    # when a relevant source statement is plainly present.
    denial_patterns = [
        "does not provide information",
        "no information is available",
        "paper does not discuss",
    ]

    if any(p in answer.lower() for p in denial_patterns):
        issues.append(
            "Answer denies available information — "
            "requires source review"
        )

    return {
        "passes_basic_checks": len(issues) == 0,
        "requested_experiments": requested,
        "mentioned_experiments": mentioned,
        "issues": issues,
        "scientifically_verified": False
    }


for test_id in ["H07_SAFFIRE_THICK", "H10_PREVENT_VS_EXTINGUISH"]:
    test = next(t for t in v32_eval if t["id"] == test_id)
    result = next(r for r in v4_results if r["id"] == test_id)

    audit = generic_coverage_check(
        test["question"],
        result["answer"],
        test["evidence"]
    )

    print("\n", test_id)
    print(audit)


In [ ]:

import fitz
import copy
from pathlib import Path

PAPERS = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/papers"
)

def expand_adjacent_pages(test, before=1, after=0):
    """Include neighboring PDF pages without using reference answers."""
    expanded = copy.deepcopy(test)
    seen = {
        (e["filename"], e["pdf_page"])
        for e in test["evidence"]
    }

    extra = []

    for item in test["evidence"]:
        path = PAPERS / item["filename"]

        with fitz.open(path) as pdf:
            for page in range(
                max(1, item["pdf_page"] - before),
                min(len(pdf), item["pdf_page"] + after) + 1
            ):
                key = (item["filename"], page)

                if key in seen:
                    continue

                seen.add(key)

                # Keep the same document identifier.
                source = item["citation"].split(":")[0]

                extra.append({
                    "citation": f"{source}:pdf-p{page}",
                    "filename": item["filename"],
                    "pdf_page": page,
                    "text": " ".join(
                        pdf[page - 1].get_text().split()
                    )
                })

    expanded["evidence"] = extra + expanded["evidence"]

    return expanded


h07_expanded = expand_adjacent_pages(h07)

print("Expanded evidence pages:")
for e in h07_expanded["evidence"]:
    print(e["citation"])


In [ ]:

evidence_text = "\n\n".join(
    f"[{e['citation']}]\n{e['text']}"
    for e in h07_expanded["evidence"]
)

messages = [
    {
        "role": "system",
        "content": (
            "You are FireSafe AI. Answer only from NASA evidence. "
            "Address every named experiment and every requested "
            "measurement. Preserve units, times and conditions. "
            "Do not invent missing findings. Cite supplied source IDs."
        )
    },
    {
        "role": "user",
        "content": (
            f"Question: {h07_expanded['question']}\n\n"
            f"NASA evidence:\n{evidence_text}"
        )
    }
]

answer = generate_v4(messages, max_new_tokens=350)

print("FIRESAFE V4.2:\n")
print(answer)

print("\nQUESTION-DRIVEN COVERAGE:")
print(
    generic_coverage_check(
        h07_expanded["question"],
        answer,
        h07_expanded["evidence"]
    )
)


In [ ]:

import re
import fitz
from pathlib import Path

saffire_pdf = Path(
    "/kaggle/input/datasets/arifengine/"
    "firesafe-ai-v3-training-data/FireSafe_AI/v3/papers/"
    "saffire_research.pdf"
)

with fitz.open(saffire_pdf) as pdf:
    p8 = " ".join(pdf[7].get_text().split())
    p9 = " ".join(pdf[8].get_text().split())

# Page 8 identifies Saffire IV-2.
# Page 9 continues its sentence with the result.
iv_identity = re.search(
    r"For Saffire IV-2, the two-sided PMMA sample flame",
    p8, re.I
)

iv_result = re.search(
    r"burning in air grew to\s*([\d.]+)\s*cm "
    r"after\s*(\d+)\s*seconds",
    p9, re.I
)

v_result = re.search(
    r"In Saffire V-3,.*?in\s*([\d.]+)% oxygen"
    r".*?grew to\s*([\d.]+)\s*cm "
    r"after\s*(\d+)\s*seconds",
    p9, re.I
)

assert iv_identity and iv_result and v_result, (
    "Could not establish source-backed measurements."
)

source_facts = {
    "Saffire IV-2": {
        "length_cm": float(iv_result.group(1)),
        "time_s": int(iv_result.group(2)),
        "sources": ["SAFFIRE:pdf-p8", "SAFFIRE:pdf-p9"]
    },
    "Saffire V-3": {
        "length_cm": float(v_result.group(2)),
        "time_s": int(v_result.group(3)),
        "oxygen_pct": float(v_result.group(1)),
        "sources": ["SAFFIRE:pdf-p9"]
    }
}

print("SOURCE-VERIFIED FACTS:")
for name, facts in source_facts.items():
    print(name, facts)

# Check the model's answer for each experiment-specific length.
model_answer = answer

for name, facts in source_facts.items():
    flight = "IV" if "IV" in name else "V"

    match = re.search(
        rf"Saffire\s+{flight}(?:-\d+)?"
        rf".*?(\d+(?:\.\d+)?)\s*cm",
        model_answer,
        flags=re.I | re.S
    )

    if not match:
        print(f"⚠️ {name}: measurement missing")
        continue

    reported = float(match.group(1))
    expected = facts["length_cm"]

    if reported == expected:
        print(f"✅ {name}: correct flame length")
    else:
        print(
            f"❌ {name}: model said {reported} cm, "
            f"NASA source says {expected} cm"
        )


In [ ]:

import json
import re

INTERPRET_SYSTEM = """You are FireSafe AI, a NASA combustion
research assistant.

The software has already displayed verified experimental measurements.

Your task is ONLY to explain the scientific limitations
of comparing the supplied experiments.

Rules:
- Do not repeat numerical values or measurements.
- Do not claim a measurement is missing.
- Do not invent causal explanations.
- Distinguish observations from controlled comparisons.
- Explain why different configurations and atmospheres
  limit direct causal conclusions.
- Keep the explanation to 1-2 sentences.
"""

messages = [
    {"role": "system", "content": INTERPRET_SYSTEM},
    {
        "role": "user",
        "content": (
            "Explain the limitations of comparing these "
            "two NASA experiments:\n\n"
            + json.dumps(source_facts, indent=2)
        )
    }
]

interpretation = generate_v4(
    messages,
    max_new_tokens=120
)

print("AI INTERPRETATION:\n")
print(interpretation)


In [ ]:

import copy
import json

# Keep the original NASA records unchanged.
evidence_v45 = copy.deepcopy(verified_facts_v44)

# Background reference information, separate from NASA evidence.
background_knowledge = {
    "standard_dry_air_oxygen_pct": {
        "value": 20.95,
        "unit": "%",
        "description": "Approximate oxygen volume fraction in standard dry air",
        "provenance": "general_scientific_reference",
        "assumption": (
            "The reported air atmosphere has approximately "
            "standard dry-air composition."
        )
    }
}

def compare_oxygen(facts, background):
    iv = facts["Saffire IV-2"]
    v = facts["Saffire V-3"]

    assert iv["atmosphere"].lower() == "air"
    assert isinstance(v["oxygen_pct"], (int, float))

    reference = background["standard_dry_air_oxygen_pct"]
    baseline = reference["value"]

    return {
        "claim": (
            "Saffire V-3 had a higher oxygen concentration "
            "than Saffire IV-2 under the standard-air assumption."
        ),
        "evidence_type": "conditional_inference",
        "nasa_evidence": [
            "Saffire IV-2.atmosphere",
            "Saffire V-3.oxygen_pct"
        ],
        "background_assumption": reference["assumption"],
        "comparison": {
            "assumed_iv_oxygen_pct": baseline,
            "reported_v_oxygen_pct": v["oxygen_pct"],
            "v_higher_than_iv": v["oxygen_pct"] > baseline
        },
        "limitation": (
            "Different sample geometries and atmospheric "
            "conditions prevent an oxygen-only causal conclusion."
        )
    }

oxygen_inference = compare_oxygen(
    evidence_v45,
    background_knowledge
)

print(json.dumps(
    oxygen_inference,
    indent=2,
    ensure_ascii=False
))


In [ ]:

# Build a source-grounded comparison from exact PDF passages.
# This demonstration selects known relevant pages;
# future versions should select these using retrieval.

import re

source_selection = {
    "BASS": [22],
    "SAFFIRE": [8, 10]
}

def evidence_snippet(source, page, term, window=300):
    text = pages[(source, page)]
    match = re.search(re.escape(term), text, re.I)

    if not match:
        return None

    start = max(0, match.start() - 60)
    end = min(len(text), match.end() + window)

    return {
        "source_id": f"{source}:pdf-p{page}",
        "quote": text[start:end].strip()
    }

source_records = [
    evidence_snippet(
        "BASS", 22,
        "Results: Thin Fuel Flame Spread and Extinction",
        650
    ),
    evidence_snippet(
        "SAFFIRE", 8,
        "concurrent flames spreading over thin fuels",
        250
    ),
    evidence_snippet(
        "SAFFIRE", 8,
        "Both concurrent samples (IV-2 and V-3)",
        350
    ),
    evidence_snippet(
        "SAFFIRE", 10,
        "strong influence of the flow characteristics",
        450
    )
]

source_records = [
    record for record in source_records
    if record is not None
]

for record in source_records:
    print("\n" + "=" * 60)
    print("SOURCE:", record["source_id"])
    print("EXACT PDF PASSAGE:")
    print(record["quote"])

assert len(source_records) == 4, (
    "A source passage was not located. "
    "Check the PDF extraction before continuing."
)

print("\n✅ Four evidence excerpts located")


In [ ]:

import json

EVIDENCE_FIRST_SYSTEM = """You are FireSafe AI, a NASA combustion
research assistant.

The application will display the original NASA evidence separately.

Your task is to explain what can and cannot be concluded when
comparing BASS and Saffire.

Rules:
- Use only the supplied evidence.
- Do not reproduce numerical measurements.
- Do not claim one study has faster flames than the other.
- Do not invent experimental conditions.
- Distinguish flame spread from flame growth and fuel regression.
- Do not introduce additional scientific mechanisms.
- Explain limitations in at most two sentences.
"""

messages = [
    {
        "role": "system",
        "content": EVIDENCE_FIRST_SYSTEM
    },
    {
        "role": "user",
        "content": (
            "Question: What do BASS and Saffire reveal about "
            "flame behavior in microgravity, and what limits "
            "a direct comparison?\n\n"
            "NASA evidence:\n"
            + json.dumps(source_records, indent=2)
        )
    }
]

interpretation = generate_firesafe(
    messages,
    max_new_tokens=160
)

print("FIRESAFE AI INTERPRETATION:\n")
print(interpretation)


In [ ]:

print("=" * 70)
print("FIRESAFE AI — EVIDENCE-FIRST RESEARCH ANSWER")
print("=" * 70)

print("\nNASA EVIDENCE:")
for record in source_records:
    print(f"\n[{record['source_id']}]")
    print(record["quote"])

print("\nAI INTERPRETATION (REVIEW-PENDING):")
print(interpretation)


In [ ]:

import re
import numpy as np
from collections import defaultdict

def firesafe_retrieve(question, pages_per_study=2):
    """Rank actual PDF pages, respecting named studies."""
    requested = requested_sources(question)

    q = vectorizer.transform([question])
    scores = (matrix @ q.T).toarray().ravel()

    # Keep each page's best-matching passage score.
    page_scores = {}

    for i, passage in enumerate(passages):
        source = passage["source"]
        page = passage["page"]

        if requested and source not in requested:
            continue

        key = (source, page)
        page_scores[key] = max(
            page_scores.get(key, 0.0),
            float(scores[i])
        )

    ranked = sorted(
        page_scores,
        key=page_scores.get,
        reverse=True
    )

    selected = []
    counts = defaultdict(int)

    if requested:
        # Give each named study its own evidence budget.
        for source in requested:
            for key in ranked:
                if key[0] != source:
                    continue
                if page_scores[key] <= 0:
                    continue
                if counts[source] >= pages_per_study:
                    break

                selected.append(key)
                counts[source] += 1
    else:
        selected = [
            key for key in ranked
            if page_scores[key] > 0
        ][:pages_per_study * 2]

    return selected


def firesafe_extract(question, source, page, max_chars=1400):
    """Select relevant complete-ish sentences from a PDF page."""
    text = pages[(source, page)]
    sentences = clean_sentences(text)

    if not sentences:
        return None

    # Rank sentences by relevance to the question.
    qvec = vectorizer.transform([question])
    svec = vectorizer.transform(sentences)
    relevance = (svec @ qvec.T).toarray().ravel()

    # Include neighboring sentences for context.
    best = np.argsort(relevance)[::-1][:2]
    indices = set()

    for i in best:
        i = int(i)
        if relevance[i] <= 0:
            continue

        for j in range(
            max(0, i - 1),
            min(len(sentences), i + 2)
        ):
            indices.add(j)

    if not indices:
        return None

    chosen = " ".join(
        sentences[i] for i in sorted(indices)
    )

    # Preserve PDF extraction; do not rewrite claims.
    if len(chosen) > max_chars:
        chosen = chosen[:max_chars].rsplit(" ", 1)[0]
        chosen += " [excerpt truncated]"

    return {
        "source_id": f"{source}:pdf-p{page}",
        "source": source,
        "page": page,
        "text": chosen
    }


def ask_firesafe_v5(question):
    selected_pages = firesafe_retrieve(question)

    records = []
    for source, page in selected_pages:
        record = firesafe_extract(
            question, source, page
        )
        if record:
            records.append(record)

    if not records:
        return {
            "answer": "Insufficient retrieved evidence.",
            "evidence": [],
            "status": "insufficient_evidence"
        }

    # Evidence stays grouped by its source.
    grouped = defaultdict(list)
    for record in records:
        grouped[record["source"]].append(record)

    evidence_blocks = []

    for source, items in grouped.items():
        block = f"STUDY: {source}\n"

        for item in items:
            block += (
                f"\n[{item['source_id']}]\n"
                f"{item['text']}\n"
            )

        evidence_blocks.append(block)

    prompt = "\n\n".join(evidence_blocks)

    messages = [
        {
            "role": "system",
            "content": """You are FireSafe AI.

Interpret only the supplied NASA evidence.

Rules:
- Keep each observation with its named study.
- Do not transfer measurements between studies.
- Do not introduce new numbers or conditions.
- Do not claim a controlled comparison without evidence.
- Distinguish flame spread, flame growth, and regression.
- Preserve uncertainty and limitations.
- Cite exact supplied [SOURCE:pdf-pN] identifiers.
- If evidence is insufficient, say so.
- Write at most three concise sentences.

Your interpretation will be labeled REVIEW-PENDING."""
        },
        {
            "role": "user",
            "content": (
                f"QUESTION:\n{question}\n\n"
                f"NASA EVIDENCE:\n{prompt}"
            )
        }
    ]

    answer = generate_firesafe(
        messages,
        max_new_tokens=210
    )

    allowed = {r["source_id"] for r in records}
    cited = set(re.findall(
        r"\[([A-Za-z0-9_-]+:pdf-p\d+)\]",
        answer
    ))

    warnings = []

    if not cited:
        warnings.append("Interpretation lacks PDF citations")

    if cited - allowed:
        warnings.append(
            "Interpretation includes citations not retrieved"
        )

    # Check whether introduced numeric tokens occur in evidence.
    numeric_pattern = r"(?<![A-Za-z0-9])\d+(?:\.\d+)?"
    answer_body = re.sub(r"\[[^\]]+\]", "", answer)

    answer_numbers = set(
        re.findall(numeric_pattern, answer_body)
    )
    evidence_numbers = set(re.findall(
        numeric_pattern,
        " ".join(r["text"] for r in records)
    ))

    added_numbers = sorted(
        answer_numbers - evidence_numbers
    )

    if added_numbers:
        warnings.append(
            f"Numbers not in selected evidence: {added_numbers}"
        )

    print("=" * 65)
    print("FIRESAFE AI — NASA RESEARCH")
    print("=" * 65)

    print("\nQUESTION:", question)
    print("\nRETRIEVED EVIDENCE:")

    for item in records:
        print(f"\n[{item['source_id']}]")
        print(item["text"])

    print("\nAI INTERPRETATION — REVIEW-PENDING:")
    print(answer)

    print("\nAUDIT WARNINGS:", warnings or "None")
    print("STATUS: requires_scientific_review")

    return {
        "question": question,
        "evidence": records,
        "interpretation": answer,
        "warnings": warnings,
        "status": "requires_scientific_review"
    }


print("✅ FireSafe AI v5.0 ready")


In [ ]:

questions = [
    "How does airflow affect flame spread in microgravity?",
    "What differences exist between BASS and Saffire flame behavior?",
    "What do Saffire experiments reveal about fire growth limitations?"
]

results_v5 = []

for question in questions:
    result = ask_firesafe_v5(question)
    results_v5.append(result)
    print("\n\n")


In [ ]:

import re

def firesafe_evidence_gate(result):
    """
    Minimum trust gate for FireSafe AI v5.0.

    A passing answer is NOT scientifically verified.
    A failing answer is withheld from the research response.
    """

    evidence = result.get("evidence", [])
    interpretation = result.get("interpretation", "")

    allowed_ids = {
        item["source_id"] for item in evidence
    }

    citations = set(re.findall(
        r"\[([A-Za-z0-9_-]+:pdf-p\d+)\]",
        interpretation
    ))

    errors = []

    if not citations:
        errors.append("No supporting PDF-page citations")

    unknown_citations = citations - allowed_ids

    if unknown_citations:
        errors.append(
            "Unknown citations: "
            + ", ".join(sorted(unknown_citations))
        )

    # Remove citations before checking numerical claims.
    answer_body = re.sub(
        r"\[[^\]]+\]", "",
        interpretation
    )

    number_pattern = (
        r"(?<![A-Za-z0-9])\d+(?:\.\d+)?"
    )

    answer_numbers = set(re.findall(
        number_pattern, answer_body
    ))

    evidence_text = " ".join(
        item["text"] for item in evidence
    )

    evidence_numbers = set(re.findall(
        number_pattern, evidence_text
    ))

    new_numbers = answer_numbers - evidence_numbers

    if new_numbers:
        errors.append(
            "Numbers absent from evidence: "
            + ", ".join(sorted(new_numbers))
        )

    passed = len(errors) == 0

    return {
        "question": result.get("question"),
        "status": (
            "review_pending"
            if passed else "interpretation_withheld"
        ),
        "evidence": evidence,
        "interpretation": (
            interpretation if passed else None
        ),
        "rejected_interpretation": (
            None if passed else interpretation
        ),
        "audit_errors": errors,
        "scientifically_verified": False
    }


def show_firesafe_review(result):
    checked = firesafe_evidence_gate(result)

    print("\n" + "=" * 65)
    print("FIRESAFE AI — EVIDENCE REVIEW")
    print("=" * 65)

    print("\nQUESTION:")
    print(checked["question"])

    print("\nNASA SOURCE PASSAGES:")
    for item in checked["evidence"]:
        print(f"\n[{item['source_id']}]")
        print(item["text"])

    if checked["interpretation"] is None:
        print("\nAI INTERPRETATION: WITHHELD")
        print(
            "The generated explanation did not meet "
            "the minimum evidence requirements."
        )
        print("\nREASONS:")
        for error in checked["audit_errors"]:
            print("-", error)
    else:
        print("\nAI INTERPRETATION (REVIEW-PENDING):")
        print(checked["interpretation"])
        print(
            "\nThis answer has not passed "
            "semantic scientific verification."
        )

    print("\nSTATUS:", checked["status"])
    return checked


reviews = [
    show_firesafe_review(result)
    for result in results_v5
]


In [ ]:

import json
import re

CLAIM_SYSTEM_V51 = """You are FireSafe AI, a NASA research assistant.

Return ONLY a valid JSON object:
{
  "claims": [
    {
      "claim": "A concise scientific statement",
      "source_id": "BASS:pdf-p22",
      "support_quote": "An EXACT continuous excerpt from the source",
      "claim_type": "observation"
    }
  ],
  "limitations": [
    "A limitation or uncertainty"
  ]
}

Rules:
- Each claim must refer to ONE study.
- Copy support_quote verbatim from its source.
- Only use source IDs provided in the evidence.
- Do not transfer observations between studies.
- Do not invent numbers or causal conclusions.
- Distinguish observations from hypotheses.
- If a claim requires several sources, split it into
  separate claims.
- Use at most 4 claims.
- If evidence is insufficient, return an empty claims list.
"""

def parse_claim_json(raw):
    # First try complete JSON.
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Extract a JSON object if model added wrapper text.
    match = re.search(r"\{[\s\S]*\}", raw)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def ask_firesafe_claims(question):
    selected = firesafe_retrieve(question)

    records = [
        firesafe_extract(question, source, page)
        for source, page in selected
    ]
    records = [r for r in records if r is not None]

    if not records:
        print("No relevant NASA evidence retrieved.")
        return None

    messages = [
        {"role": "system", "content": CLAIM_SYSTEM_V51},
        {
            "role": "user",
            "content": (
                f"QUESTION:\n{question}\n\n"
                "NASA EVIDENCE:\n"
                + json.dumps(records, ensure_ascii=False)
            )
        }
    ]

    raw = generate_firesafe(
        messages,
        max_new_tokens=450
    )

    parsed = parse_claim_json(raw)

    if not isinstance(parsed, dict):
        print("Invalid model JSON; no claims accepted.")
        print(raw)
        return None

    sources = {
        r["source_id"]: r["text"]
        for r in records
    }

    accepted = []
    flagged = []

    for claim in parsed.get("claims", []):
        if not isinstance(claim, dict):
            continue

        source_id = claim.get("source_id", "")
        quote = claim.get("support_quote", "")
        statement = claim.get("claim", "")

        errors = []

        if source_id not in sources:
            errors.append("Unknown source ID")
        elif not quote or quote not in sources[source_id]:
            errors.append("Quote not found verbatim")

        if not statement:
            errors.append("Missing scientific statement")

        checked = {
            **claim,
            "checks": errors
        }

        if errors:
            flagged.append(checked)
        else:
            accepted.append(checked)

    result = {
        "question": question,
        "accepted_quote_matched_claims": accepted,
        "flagged_claims": flagged,
        "limitations": parsed.get("limitations", []),
        "source_records": records,
        "scientifically_verified": False
    }

    print("\n" + "=" * 65)
    print("FIRESAFE AI — CLAIM REVIEW")
    print("=" * 65)

    print("\nQUOTE-MATCHED CLAIMS:")
    for c in accepted:
        print(f"\n- {c['claim']}")
        print(f"  Source: [{c['source_id']}]")
        print(f"  Quote: {c['support_quote']}")

    print("\nFLAGGED CLAIMS:")
    for c in flagged:
        print("-", c.get("claim"), c["checks"])

    print("\nLIMITATIONS:")
    for limitation in result["limitations"]:
        print("-", limitation)

    print("\nSCIENTIFICALLY VERIFIED: False")

    return result


result_v51 = ask_firesafe_claims(
    "How does airflow affect opposed-flow and "
    "concurrent-flow flame spread in microgravity?"
)


In [ ]:

import re

def comparison_consistency_check(claim, quote):
    issues = []

    c = claim.lower()
    q = quote.lower()

    # Detect the specific comparison structure responsible
    # for several observed FireSafe attribution errors.
    has_cross_condition_comparison = (
        bool(re.search(r"\bthan\b", c))
        and "opposed" in c
        and "concurrent" in c
    )

    if has_cross_condition_comparison:
        if not (
            "opposed" in q
            and "concurrent" in q
        ):
            issues.append(
                "Cross-flow comparison is not supported "
                "by a quote describing both flow conditions"
            )

    # Flag strong negative claims when their exact quote
    # contains no matching negative assertion.
    strong_negative = bool(re.search(
        r"\b(do not|does not|did not|never|cannot)\b",
        c
    ))

    quote_negative = bool(re.search(
        r"\b(do not|does not|did not|never|cannot|no)\b",
        q
    ))

    if strong_negative and not quote_negative:
        issues.append(
            "Negative claim is absent from its supporting quote"
        )

    return issues


def review_v52(result):
    reviewed = []
    rejected = []

    # Recheck all claims, including ones already flagged
    # by the verbatim-quotation validator.
    all_claims = (
        result["accepted_quote_matched_claims"]
        + result["flagged_claims"]
    )

    for item in all_claims:
        issues = list(item.get("checks", []))

        issues += comparison_consistency_check(
            item.get("claim", ""),
            item.get("support_quote", "")
        )

        checked = {
            **item,
            "review_issues": list(dict.fromkeys(issues))
        }

        if checked["review_issues"]:
            rejected.append(checked)
        else:
            reviewed.append(checked)

    print("\n" + "=" * 60)
    print("FIRESAFE AI v5.2 — CLAIM CONSISTENCY")
    print("=" * 60)

    print("\nQUOTE-MATCHED CANDIDATES:")
    for item in reviewed:
        print("-", item["claim"])

    print("\nFLAGGED CLAIMS:")
    for item in rejected:
        print("\n-", item["claim"])
        for issue in item["review_issues"]:
            print("  Reason:", issue)

    print("\nScientifically verified: NO")

    return {
        "candidates": reviewed,
        "flagged": rejected,
        "scientifically_verified": False
    }


review_v52_result = review_v52(result_v51)


In [ ]:

import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

NLI_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Prefer the second T4 to avoid moving your Qwen model.
nli_device = torch.device(
    "cuda:1" if torch.cuda.device_count() > 1
    else "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_NAME
).to(nli_device).eval()

labels = {
    int(k): str(v).lower()
    for k, v in nli_model.config.id2label.items()
}

def verify_entailment(quote, claim):
    # Premise = NASA evidence
    # Hypothesis = Qwen's claim
    inputs = nli_tokenizer(
        quote,
        claim,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(nli_device)

    with torch.inference_mode():
        logits = nli_model(**inputs).logits[0]
        probs = torch.softmax(logits, dim=-1).tolist()

    scores = {
        labels[i]: round(float(p), 4)
        for i, p in enumerate(probs)
    }

    # Screening threshold only; not a calibrated
    # scientific verification probability.
    decision = (
        "entailment_candidate"
        if scores.get("entailment", 0) >= 0.85
        else "needs_review"
    )

    return {
        "decision": decision,
        "scores": scores
    }


# Test ALL original v5.1 claims, including rejected ones.
all_claims = (
    result_v51["accepted_quote_matched_claims"]
    + result_v51["flagged_claims"]
)

rows = []

for item in all_claims:
    verdict = verify_entailment(
        item.get("support_quote", ""),
        item.get("claim", "")
    )

    rows.append({
        "claim": item["claim"],
        "quote_matched": not bool(item.get("checks")),
        "entailment": verdict["scores"].get("entailment", 0),
        "neutral": verdict["scores"].get("neutral", 0),
        "contradiction": verdict["scores"].get("contradiction", 0),
        "decision": verdict["decision"]
    })

nli_results = pd.DataFrame(rows)

display(nli_results)


In [ ]:

import json
from pathlib import Path

baseline = {
    "system": "FireSafe AI",
    "component": "scientific_claim_verifier",
    "benchmark": "15_claim_development_set",
    "total": 15,
    "correct_binary": 14,
    "accuracy": 14 / 15,
    "false_acceptances": 0,
    "false_rejections": 1,
    "entailment_threshold": 0.85,
    "contradiction_threshold": 0.85,
    "notes": [
        "Development set, not independent held-out evaluation",
        "Includes cases used during verifier design",
        "Only binary accept/reject behavior evaluated",
        "A review_candidate is not scientifically verified"
    ]
}

path = Path(
    "/kaggle/working/firesafe_verifier_baseline.json"
)

path.write_text(
    json.dumps(baseline, indent=2),
    encoding="utf-8"
)

print("Saved:", path)


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

label_suggestions = {
    "supported_observation": "supported",
    "contradicted_observation": "contradicted",
    "unsupported_numerical_comparison": "insufficient",
    "experiment_misattribution": "insufficient",
    "causal_or_safety_overclaim": "insufficient"
}

def relevant_excerpt(question, text, max_chars=1100):
    sentences = clean_sentences(text)
    if not sentences:
        return text[:max_chars]

    qvec = vectorizer.transform([question])
    svec = vectorizer.transform(sentences)
    scores = (svec @ qvec.T).toarray().ravel()

    best = int(np.argmax(scores))
    selected = range(
        max(0, best - 1),
        min(len(sentences), best + 2)
    )

    return " ".join(
        sentences[i] for i in selected
    )[:max_chars]


review_df = df50.copy()

review_df["suggested_label"] = (
    review_df["category"].map(label_suggestions)
)

review_df["source_excerpt"] = review_df.apply(
    lambda r: relevant_excerpt(
        r["claim"], r["evidence_text"]
    ),
    axis=1
)

review_columns = [
    "id",
    "category",
    "source_id",
    "claim",
    "suggested_label",
    "source_excerpt",
    "gold_label",
    "review_notes"
]

review_path = Path(
    "/kaggle/working/firesafe_50_claims_label_review.csv"
)

review_df[review_columns].to_csv(
    review_path, index=False
)

print("✅ Review file:", review_path)
print("✅ Candidates:", len(review_df))
print("\nSuggested label distribution:")
print(review_df["suggested_label"].value_counts())

display(review_df[
    ["id", "claim", "suggested_label", "gold_label"]
].head(10))


In [ ]:

from pathlib import Path
import pandas as pd

review_path = Path(
    "/kaggle/working/firesafe_50_claims_label_review.csv"
)

reviewed = pd.read_csv(review_path).fillna("")

valid_labels = {
    "supported",
    "contradicted",
    "insufficient"
}

remaining = reviewed[
    ~reviewed["gold_label"].isin(valid_labels)
]

print("Total:", len(reviewed))
print("Reviewed:", len(reviewed) - len(remaining))
print("Remaining:", len(remaining))

if remaining.empty:
    frozen_path = Path(
        "/kaggle/working/firesafe_50_claims_frozen.csv"
    )
    reviewed.to_csv(frozen_path, index=False)
    print("✅ Frozen labeled dataset:", frozen_path)
else:
    print("Labels still requiring review:")
    display(remaining[["id", "claim", "gold_label"]])


In [ ]:

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

REVIEW_FILE = Path(
    "/kaggle/working/firesafe_50_claims_label_review.csv"
)

review_df = pd.read_csv(REVIEW_FILE).fillna("")

VALID_LABELS = [
    "PENDING_REVIEW",
    "supported",
    "contradicted",
    "insufficient"
]

# Change this to 5, 10, 15 ... to review later batches.
batch_start = 0
batch_size = 5

controls = []

for idx in range(batch_start, min(batch_start + batch_size,
                                  len(review_df))):
    r = review_df.iloc[idx]

    print("=" * 65)
    print(f"{r['id']} | {r['source_id']}")
    print("\nCLAIM:", r["claim"])
    print("\nNASA EXCERPT:")
    print(r["source_excerpt"])
    print("\nSUGGESTED LABEL:", r["suggested_label"])

    current = r["gold_label"]
    if current not in VALID_LABELS:
        current = "PENDING_REVIEW"

    selector = widgets.Dropdown(
        options=VALID_LABELS,
        value=current,
        description="Your label:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="350px")
    )

    note = widgets.Text(
        value=str(r["review_notes"]),
        placeholder="Optional scientific review note",
        description="Notes:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="650px")
    )

    display(selector, note)
    controls.append((idx, selector, note))


save_button = widgets.Button(
    description="Save reviewed labels",
    button_style="success"
)

save_output = widgets.Output()


def save_reviews(button):
    with save_output:
        clear_output()

        for idx, selector, note in controls:
            review_df.at[idx, "gold_label"] = selector.value
            review_df.at[idx, "review_notes"] = note.value

        review_df.to_csv(REVIEW_FILE, index=False)

        completed = review_df["gold_label"].isin(
            VALID_LABELS[1:]
        ).sum()

        print(f"✅ Saved: {completed}/50 labels reviewed")
        print(f"Remaining: {50 - completed}")


save_button.on_click(save_reviews)

display(save_button, save_output)


In [ ]:

import pandas as pd

REVIEW_FILE = (
    "/kaggle/working/"
    "firesafe_50_claims_label_review.csv"
)

df = pd.read_csv(REVIEW_FILE).fillna("")

approved = [
    "FS-001", "FS-002", "FS-003",
    "FS-004", "FS-005"
]

mask = df["id"].isin(approved)

df.loc[mask, "gold_label"] = "supported"
df.loc[mask, "review_notes"] = (
    "Reviewed against the supplied NASA PDF excerpt."
)

df.to_csv(REVIEW_FILE, index=False)

print("✅ Reviewed:", mask.sum(), "claims")
print("Remaining:", (
    df["gold_label"] == "PENDING_REVIEW"
).sum())


In [ ]:

from pathlib import Path

adapter_path = Path(
    "/kaggle/working/firesafe_qwen3b_v32_adapter"
)

print("Adapter exists:", adapter_path.exists())

if adapter_path.exists():
    print("Files:")
    for file in adapter_path.iterdir():
        print("-", file.name)


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF-TOKEN")


In [ ]:

from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    for path in root.rglob("*"):
        if path.is_file() and (
            "firesafe" in path.name.lower()
            or "adapter" in path.name.lower()
            or path.suffix.lower() == ".zip"
        ):
            print(path)
